# 🎬 MovieMind – Artifact Builder (Alan Vourch Clean Dataset)

Run this notebook on **Kaggle** (CPU is sufficient). It will:
- Download the 1M movie dataset from Alan Vourch
- Extract correct directors and cast from JSON
- Build the FAISS index, processed parquet, and TF‑IDF vectorizer
- Download the final artifacts.zip

---

## 1. Install dependencies

In [1]:
!pip install -q faiss-cpu sentence-transformers pyarrow kagglehub scikit-learn
print("✅ Dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.3 MB/s eta 0:00:00:00:0100:01
✅ Dependencies installed


## 2. Download the Alan Vourch dataset
We use `kagglehub` to pull the latest snapshot and copy the CSV to a fixed path.

In [2]:
import kagglehub, os, shutil

print("Downloading dataset via kagglehub...")
path = kagglehub.dataset_download("alanvourch/tmdb-movies-daily-updates")
print("Dataset downloaded to:", path)

os.makedirs("/kaggle/working/data", exist_ok=True)
for fname in os.listdir(path):
    if fname.endswith(".csv") and "TMDB_all_movies" in fname:
        src = os.path.join(path, fname)
        dst = "/kaggle/working/data/TMDB_all_movies.csv"
        shutil.copy(src, dst)
        print(f"✅ Copied {fname} -> {dst}")
        break
else:
    print("❌ Could not find TMDB_all_movies.csv. Check the dataset contents.")

Dataset downloaded to: /kaggle/input/datasets/alanvourch/tmdb-movies-daily-updates
✅ Copied TMDB_all_movies.csv -> /kaggle/working/data/TMDB_all_movies.csv


## 3. Constants & output paths
Define all file paths and model configuration.

In [4]:
DATA_DIR = "/kaggle/working/data"
ARTIFACTS_DIR = "/kaggle/working/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TMDB_CSV = os.path.join(DATA_DIR, "TMDB_all_movies.csv")
PROCESSED_PARQUET = os.path.join(ARTIFACTS_DIR, "movies_processed_final.parquet")
FAISS_INDEX = os.path.join(ARTIFACTS_DIR, "movies_faiss.index")
TFIDF_VECTORIZER = os.path.join(ARTIFACTS_DIR, "tfidf_vectorizer.pkl")
MODEL_NAME_FILE = os.path.join(ARTIFACTS_DIR, "model_name.txt")

EMBEDDING_MODEL = "all-MiniLM-L6-v2"

FINAL_COLS = [
    "movie_id", "title", "year", "genres", "overview",
    "soup", "norm_popularity", "vote_average", "vote_count",
    "avg_rating", "popularity_log", "director", "cast", "keywords"
]

print("✅ Constants set")


✅ Constants set


## 4. Helper functions (cleaning, JSON‑aware cast parser)
These handle TMDB’s JSON structures for **cast**, **genres**, and **keywords**, and normalise titles.

In [5]:
import pandas as pd, numpy as np, re, ast, time

def clean_title(title):
    if pd.isna(title): return ""
    return re.sub(r'\s*\(\d{4}\)\s*$', '', str(title)).strip()

def normalize_title(title):
    if pd.isna(title): return ""
    title = str(title).strip()
    match = re.match(r'^(.*),\s*(The|A|An)$', title, re.IGNORECASE)
    if match:
        return f"{match.group(2)} {match.group(1)}"
    return title

def clean_director(val):
    """Already a plain string in this dataset; just strip."""
    if pd.isna(val): return ""
    return str(val).strip()

def clean_cast(val, top_n=5):
    """Parse cast from JSON or plain comma‑separated, return space‑separated top N names."""
    if pd.isna(val) or not val:
        return ""
    val_str = str(val).strip()
    names = []
    # Try JSON first (TMDB style: [{"name":"...","character":"..."}, ...])
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            items = ast.literal_eval(val_str)
            for item in items:
                if isinstance(item, dict):
                    names.append(item.get("name", ""))
                else:
                    names.append(str(item))
        except:
            pass
    # If still empty, treat as comma‑separated
    if not names:
        names = [name.strip() for name in val_str.split(",")]
    # Filter empty and keep top_n
    names = [n for n in names if n]
    return " ".join(names[:top_n])

def clean_genres(val):
    """Handle JSON or plain genres string."""
    if pd.isna(val) or not val:
        return ""
    val_str = str(val).strip()
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            items = ast.literal_eval(val_str)
            names = []
            for item in items:
                if isinstance(item, dict):
                    names.append(item.get("name", ""))
                else:
                    names.append(str(item))
            return " ".join(names)
        except:
            pass
    return val_str

def clean_keywords(val):
    """Handle JSON or plain keywords string."""
    if pd.isna(val) or not val:
        return ""
    val_str = str(val).strip()
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            items = ast.literal_eval(val_str)
            names = [item.get("name", str(item)) if isinstance(item, dict) else str(item) for item in items]
            return " ".join(names)
        except:
            pass
    return val_str

## 5. Load dataset & preprocess
- Lowercases column names, fills missing columns
- Applies all cleaning functions
- Deduplicates by title+year
- Builds the semantic **soup**
- Saves processed parquet

In [7]:
print("Loading Alan Vourch TMDB dataset...")
df = pd.read_csv(TMDB_CSV, low_memory=False)
df.columns = [c.strip().lower() for c in df.columns]
print(f"Loaded {len(df)} rows. Columns: {list(df.columns)}")

# Ensure required columns exist
required_cols = ["id", "title", "overview", "release_date", "popularity",
                 "vote_average", "vote_count", "genres", "keywords", "cast", "director"]
for col in required_cols:
    if col not in df.columns:
        df[col] = ""

# Rename id → movie_id
df.rename(columns={"id": "movie_id"}, inplace=True)

# Title cleaning
df["title"] = df["title"].apply(clean_title).apply(normalize_title)

# Year extraction
if "release_date" in df.columns:
    df["year"] = pd.to_datetime(df["release_date"], errors='coerce').dt.year
else:
    df["year"] = 2000
df["year"] = df["year"].fillna(2000).astype(int)

# Text fields
df["overview"]   = df["overview"].fillna("")
df["keywords"]   = df["keywords"].apply(clean_keywords)
df["vote_average"] = pd.to_numeric(df["vote_average"], errors='coerce').fillna(0)
df["vote_count"]   = pd.to_numeric(df["vote_count"], errors='coerce').fillna(0)
df["popularity"]   = pd.to_numeric(df["popularity"], errors='coerce').fillna(0)

# Director (plain text)
df["director"] = df["director"].apply(clean_director)

# Cast (JSON‑aware parser – this fixes missing Tom Cruise films)
df["cast"] = df["cast"].apply(clean_cast, top_n=5)

# Genres (JSON‑aware)
df["genres"] = df["genres"].apply(clean_genres)

# Deduplicate by title+year
df = df.sort_values(by="vote_count", ascending=False)
df = df.drop_duplicates(subset=["title", "year"], keep="first")

# Build semantic soup (director first for priority)
df["soup"] = (
    df["director"].fillna("").str.lower() + " " +
    df["title"].fillna("").str.lower() + " " +
    df["cast"].fillna("").str.lower() + " " +
    df["genres"].fillna("").str.lower() + " " +
    df["overview"].fillna("").str.lower() + " " +
    df["keywords"].fillna("").str.lower()
)

df["avg_rating"] = df["vote_average"]
df["popularity_log"] = np.log1p(df["vote_count"])
max_log = df["popularity_log"].max() + 1e-8
min_log = df["popularity_log"].min()
df["norm_popularity"] = (df["popularity_log"] - min_log) / (max_log - min_log)

# Keep exactly the columns the predictor expects
available_final = [c for c in FINAL_COLS if c in df.columns]
df_final = df[available_final].copy()

print(f"Preprocessed: {df_final.shape}")
print(f"Movies with director: {df_final[df_final['director'] != ''].shape[0]}")
print("Sample directors:", df_final[df_final['director'] != '']['director'].head(10).tolist())
print("Sample cast:", df_final[df_final['cast'] != '']['cast'].head(5).tolist())

# Save parquet
df_final.to_parquet(PROCESSED_PARQUET, index=False)
print("✅ Parquet saved")

Loading Alan Vourch TMDB dataset...
Loaded 1196164 rows. Columns: ['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'cast', 'director', 'director_of_photography', 'writers', 'producers', 'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path']
Preprocessed: (1165963, 14)
Movies with director: 976754
Sample directors: ['Christopher Nolan', 'Christopher Nolan', 'Joss Whedon', 'Christopher Nolan', 'James Cameron', 'Tim Miller', 'David Fincher', 'Joe Russo, Anthony Russo', 'Frank Darabont', 'Quentin Tarantino']
Sample cast: ['Michael Caine Matthew McConaughey Wes Bentley Liam Dickinson Flora Nolan', 'Shannon Welles Miranda Nolan Michael Caine Andrew Pleavin Joseph Gordon-Levitt', 'Robert P. Thitoff Nate Paige Chris Hemsworth Sandra Weston Andrea-Nichole Olivas', 'Ronan 

## 6. Pre‑embedding verification
Before wasting time encoding, we check:
- Director column is plain text (no JSON)
- Martin Scorsese has no false positives (e.g., *Toy Story 3*)
- Cast contains Tom Cruise (should be many)

In [7]:
print("=" * 60)
print("Running pre‑embedding data checks...")
print("=" * 60)

director_filled = df_final['director'].notna() & (df_final['director'] != '')
num_directors = director_filled.sum()
print(f"✅ Movies with director: {num_directors} / {len(df_final)}")

sample_directors = df_final.loc[director_filled, 'director'].head(10).tolist()
print(f"✅ Sample directors: {sample_directors}")

json_like = df_final['director'].str.contains(r'^\s*\[', na=False).sum()
if json_like > 0:
    print(f"❌ WARNING: {json_like} director entries appear to be JSON strings.")
else:
    print("✅ Director column is plain text (no JSON noise).")

cast_filled = df_final['cast'].notna() & (df_final['cast'] != '')
num_cast = cast_filled.sum()
print(f"✅ Movies with cast: {num_cast} / {len(df_final)}")
sample_cast = df_final.loc[cast_filled, 'cast'].head(5).tolist()
print(f"✅ Sample cast strings: {sample_cast}")

# Check Martin Scorsese
test_director = "Martin Scorsese"
scorsese_movies = df_final[df_final['director'].str.lower() == test_director.lower()]
scorsese_titles = scorsese_movies['title'].head(15).tolist()
print(f"🎬 Movies attributed to '{test_director}': {len(scorsese_movies)}")
print(f"   Sample titles: {scorsese_titles}")

false_positives = {'Toy Story 3', 'Fury', '8 Mile', 'Tomorrowland', 'The Nun'}
actual_false = set(scorsese_titles) & false_positives
if actual_false:
    print(f"❌ FAILED: Found known false positives under '{test_director}': {actual_false}")
    print("   The director column is STILL corrupted. DO NOT proceed with embeddings.")
else:
    print(f"✅ PASSED: No known false positives found under '{test_director}'.")

# Check Tom Cruise
test_actor = "Tom Cruise"
cruise_movies = df_final[df_final['cast'].str.contains(test_actor, case=False, na=False)]
print(f"🎬 Movies containing '{test_actor}' in cast: {len(cruise_movies)}")
print(f"   Sample titles: {cruise_movies['title'].head(10).tolist()}")

if num_directors < 100000:
    print("❌ FAILED: Too few directors. The mapping is likely wrong.")
elif actual_false:
    print("❌ FAILED: False positives present. Fix the column mapping and re‑run.")
else:
    print("✅ ALL CHECKS PASSED. You can safely run the embeddings cell now.")

Running pre‑embedding data checks...
✅ Movies with director: 975471 / 1164411
✅ Sample directors: ['Christopher Nolan', 'Christopher Nolan', 'Joss Whedon', 'Christopher Nolan', 'James Cameron', 'Tim Miller', 'David Fincher', 'Joe Russo, Anthony Russo', 'Frank Darabont', 'Quentin Tarantino']
❌ WARNING: 1 director entries appear to be JSON strings.
✅ Movies with cast: 796970 / 1164411
✅ Sample cast strings: ['Flora Nolan Topher Grace Collette Wolfe Brooke Smith Benjamin Hardy', 'Angela Nathenson Adam Cole Carl Gilliard Earl Cameron Yuji Okumoto', 'Robert P. Thitoff Nate Paige Chris Hemsworth Sandra Weston Andrea-Nichole Olivas', 'Don Kress Michael Vieau James Farruggio Nicky Katt Lanny Lutz', 'Michelle Rodriguez Kevin Dorman Kelson Henderson Ilram Choi Scott Lawrence']
🎬 Movies attributed to 'Martin Scorsese': 56
   Sample titles: ['The Wolf of Wall Street', 'Shutter Island', 'The Departed', 'GoodFellas', 'Taxi Driver', 'Hugo', 'The Irishman', 'Gangs of New York', 'Casino', 'The Aviator'

## 7. Build Sentence‑Transformer embeddings & FAISS index
Encodes ~1.1 million soups on CPU (takes ~1‑2 hours).
Also creates a lightweight TF‑IDF vectorizer as backup.

In [8]:
from sentence_transformers import SentenceTransformer
import faiss, pickle

print(f"Loading model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)

soups = df_final['soup'].tolist()
print(f"Encoding {len(soups)} soups on CPU...")
start = time.time()
embeddings = model.encode(soups, show_progress_bar=True, convert_to_numpy=True, batch_size=64)
elapsed = time.time() - start
print(f"Encoded in {elapsed:.0f}s ({len(soups)/elapsed:.1f} sentences/sec)")

embeddings_f32 = np.ascontiguousarray(embeddings.astype('float32'))
dim = embeddings_f32.shape[1]
index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(embeddings_f32)
index.add(embeddings_f32)
faiss.write_index(index, FAISS_INDEX)
print("✅ FAISS index saved")

# TF‑IDF vectorizer (optional)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf.fit_transform(df_final['soup'])
with open(TFIDF_VECTORIZER, 'wb') as f:
    pickle.dump(tfidf, f)
print("✅ TF‑IDF vectorizer saved")

# Model name
with open(MODEL_NAME_FILE, 'w') as f:
    f.write(EMBEDDING_MODEL)
print("✅ All artifacts built.")

Loading model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 1165963 soups on CPU...


Batches:   0%|          | 0/18219 [00:00<?, ?it/s]

Encoded in 1048s (1112.7 sentences/sec)
✅ FAISS index saved
✅ TF‑IDF vectorizer saved
✅ All artifacts built.


## 8. Zip and download artifacts
Creates `artifacts.zip` and provides a download link.

In [9]:
!cd /kaggle/working && zip -r artifacts.zip artifacts
from IPython.display import FileLink
FileLink("artifacts.zip")

  adding: artifacts/ (stored 0%)
  adding: artifacts/model_name.txt (stored 0%)
  adding: artifacts/movies_processed_final.parquet (deflated 14%)
  adding: artifacts/movies_faiss.index (deflated 7%)
  adding: artifacts/tfidf_vectorizer.pkl (deflated 61%)


/kaggle/working/artifacts.zip